# SenGuard-MSC · Notebook 08 — Score forecast: independent test on cohorts never used

The gene-level virtual passage failed its gate in notebook 07, but the score forecast passed that one criterion. Passing one criterion out of several, after seeing it, can't justify a claim. So the forecast is frozen here and tested on cohorts that nothing in this project has touched.

In [ ]:
AMENDMENT3 = """# Analysis protocol amendment 3: independent test of the score forecast

Fixed on 2026-11-01, after notebook 07, before any cohort below was scored.

## Frozen model
virtual_passage_model.json from notebook 07, model "final": per-gene drift across 8 cohorts, 4 significant digits. Forecast of the SenCat rank score at a later passage = rank score of (earliest profile + drift x change in log2(passage + 1)). No refitting.

## Independent cohorts, chosen from GEO metadata alone
- GSE39035: bone marrow, 8 donors, P4 and P8, array.
- GSE139073: bone marrow, 6 donors with both P3 and P6 at rest (BM126, BM127, BM285, BM293, BM324, BM784), RNA-seq. Stimulated samples are not used.
- GSE9593: bone marrow, 3 donors, P2 to P7, P8 or P11, array.
If a cohort can't be parsed, it is reported as failed and the rule applies to what remains, with at least 2 cohorts.

## Gate for the interface
All must hold:
1. Absolute error of the forecast is lower than carry-forward in at least 70% of donors, pooled.
2. Median error is lower in every available cohort except at most one.
3. Negative control: drift learned from passages shuffled within donors beats carry-forward in at most 60% of donors, on average.
4. Prediction range: the 80% range set from leave-one-cohort-out errors in notebook 07 contains the actual score in at least 70% of donors.

## What this claims if it passes
A forecast of how a transcriptomic signature moves with passage. Not senescence confirmation, growth capacity, potency or safety.
"""
(REPORTS / 'ANALYSIS_AMENDMENT_3.md').write_text(AMENDMENT3, encoding='utf-8')
print('Amendment 3 sha256', sg.sha256(REPORTS / 'ANALYSIS_AMENDMENT_3.md'))
model = json.load(open(RESULTS / 'virtual_passage_model.json'))
print('Frozen model sha256', sg.sha256(RESULTS / 'virtual_passage_model.json'))
units_dev = pd.read_csv(RESULTS / 'virtual_passage_units.csv')
RANGE80 = float(np.percentile(units_dev.err_score_model, 80))
print('80% error range from leave-one-cohort-out:', round(RANGE80, 4), 'on', len(units_dev), 'donors')
GENES = model['genes']; BETA = np.array(model['beta']['final']); SIG = pd.Series(model['signature'])

## Load the independent cohorts

In [ ]:
TEST, LOG = {}, []
def attempt(key, fn):
    try:
        fn()
    except Exception as e:
        LOG.append({'cohort': key, 'status': f'failed: {type(e).__name__}: {str(e)[:240]}'})
        print(key, 'FAILED', type(e).__name__, str(e)[:300])

def add_test(key, sym, meta, kind, note=''):
    meta = meta.loc[sym.index]
    TEST[key] = dict(sym=sym, meta=meta, kind=kind)
    LOG.append({'cohort': key, 'status': 'loaded', 'samples': len(sym), 'genes': sym.shape[1], 'donors': meta.subject.nunique(), 'kind': kind, 'note': note})
    print(key, sym.shape, 'donors', meta.subject.nunique(), sorted(meta.passage.unique()))

def _39035():
    expr, m = parse_matrix(fetch(f'{sdir("GSE39035")}/matrix/GSE39035_series_matrix.txt.gz'))
    gpl = re.search(r'GPL\d+', gzip.open(RAWX / 'GSE39035_series_matrix.txt.gz', 'rt', errors='replace').read(200000)).group(0)
    sym = probe_symbols(gpl)
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    meta = pd.DataFrame({'subject': m['subject id'].astype(str), 'passage': m['passage'].astype(float)}, index=m.index)
    add_test('GSE39035', e.fillna(e.median()).fillna(0), meta, 'log_array', f'Platform {gpl}')
attempt('GSE39035', _39035)

def _139073():
    t = pd.read_csv(fetch(f'{sdir("GSE139073")}/suppl/GSE139073_raw_counts.txt.gz'), sep='\t', low_memory=False)
    print('  columns', list(t.columns)[:6], '| first ids', t.iloc[:2, 0].tolist())
    g = geo_samples('GSE139073')
    g['s'] = g.title.str.extract(r'(S\d+)')[0]
    donors = ['BM126', 'BM127', 'BM285', 'BM293', 'BM324', 'BM784']
    g = g[g.donor.isin(donors) & (g.stimulation == 'REST') & g.passage.isin(['P3', 'P6'])]
    print('  selected', g[['donor', 'passage', 's']].values.tolist())
    if isinstance(t.index, pd.RangeIndex):   # this file's header is one field shorter than its rows, so ids are already the index
        t = t.set_index(t.columns[0])
    t = t[~t.index.duplicated()]
    print('  gene ids', t.index[:3].tolist(), '| rows', len(t))
    cols = {}
    for r in g.itertuples():
        hit = [c for c in t.columns if re.search(rf'(^|[^0-9]){r.s[1:]}([^0-9]|$)', str(c))]
        if len(hit) != 1:
            raise RuntimeError(f'{r.s}: matched {hit}; columns {list(t.columns)[:8]}')
        cols[hit[0]] = r.s
    x = t[list(cols)].rename(columns=cols).apply(pd.to_numeric, errors='coerce').fillna(0).T
    meta = pd.DataFrame({'subject': g.donor.to_numpy(), 'passage': g.passage.str[1:].astype(float).to_numpy()}, index=g.s.to_numpy())
    add_test('GSE139073', to_sym(x).fillna(0), meta, 'counts', 'Resting samples')
attempt('GSE139073', _139073)

def _9593():
    expr, m = parse_matrix(fetch(f'{sdir("GSE9593")}/matrix/GSE9593_series_matrix.txt.gz'))
    gpl = re.search(r'GPL\d+', gzip.open(RAWX / 'GSE9593_series_matrix.txt.gz', 'rt', errors='replace').read(200000)).group(0)
    sym = probe_symbols(gpl)
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    t = m.title.str.extract(r'donor(\d)')[0]
    p = m.title.str.extract(r'-P(\d+)')[0].astype(float)
    meta = pd.DataFrame({'subject': 'donor' + t, 'passage': p}, index=m.index)
    add_test('GSE9593', e.fillna(e.median()).fillna(0), meta, 'array', f'Platform {gpl}')
attempt('GSE9593', _9593)
pd.DataFrame(LOG).to_csv(RESULTS / 'forecast_test_load_log.csv', index=False)
print(pd.DataFrame(LOG).to_string(index=False, max_colwidth=90))

## Test [measured]

In [ ]:
tlog = lambda p: np.log2(np.asarray(p, float) + 1)
def rank_form(sym):
    use = [g for g in GENES if g in sym.columns]
    return sym[use].rank(axis=1, pct=True), use

def evaluate(beta_full):
    out = []
    bs = pd.Series(beta_full, index=GENES)
    for key, c in TEST.items():
        R, use = rank_form(c['sym'])
        sig = SIG.reindex(use).dropna()
        s_idx = [use.index(g) for g in sig.index]
        wr = stats.rankdata(sig.to_numpy())
        score = lambda x: float(np.corrcoef(stats.rankdata(np.asarray(x)[s_idx]), wr)[0, 1])
        b = bs[use].to_numpy()
        d = R.join(c['meta']).groupby(['subject', 'passage']).mean(numeric_only=True).reset_index()
        for sub, g in d.groupby('subject'):
            g = g.sort_values('passage')
            if g.passage.nunique() < 2:
                continue
            X = g[use].to_numpy(float)
            x0, xL = X[0], X[-1]
            dt = tlog(g.passage.iloc[-1]) - tlog(g.passage.iloc[0])
            s0, sT, sF = score(x0), score(xL), score(x0 + b * dt)
            out.append({'cohort': key, 'donor': sub, 'from': g.passage.iloc[0], 'to': g.passage.iloc[-1], 'score_start': s0, 'score_actual': sT, 'score_forecast': sF, 'err_forecast': abs(sF - sT), 'err_carry': abs(s0 - sT), 'genes': len(use), 'signature_genes': len(s_idx)})
    return pd.DataFrame(out)

res = evaluate(BETA)
res['forecast_better'] = res.err_forecast < res.err_carry
res['in_range'] = res.err_forecast <= RANGE80
res['actual_rose'] = res.score_actual > res.score_start
res.to_csv(RESULTS / 'forecast_test_donors.csv', index=False)
print(res.round(4).to_string(index=False))
# Very small errors are checked before they are believed: leakage would show up as a cohort shared with training, or as identical profiles
print('Test cohorts also used in training:', sorted(set(TEST) & {'GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362'}))
for k, c in TEST.items():
    R, use = rank_form(c['sym'])
    print(k, 'correlation of each sample with the training drift vector, spread:', pd.Series(R.sub(R.mean()).to_numpy() @ pd.Series(BETA, index=GENES)[use].to_numpy()).round(1).describe()[['min', 'max']].to_dict())
    print(k, 'smallest change between two samples of one donor, mean absolute rank:', round(float(min(np.abs(R.iloc[i] - R.iloc[j]).mean() for i in range(len(R)) for j in range(i + 1, len(R)))), 4))
by = res.groupby('cohort').agg(donors=('donor', 'size'), err_forecast=('err_forecast', 'median'), err_carry=('err_carry', 'median'), better=('forecast_better', 'sum'), in_range=('in_range', 'sum'), rose=('actual_rose', 'sum'))
print(by.round(3).to_string())

# Negative control: drift learned from training cohorts with passages shuffled inside donors
TRAIN = ['GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362']
TR = {}
for k in TRAIN:
    if k not in COHORTS:
        continue
    c = COHORTS[k]
    use = [g for g in GENES if g in c['sym'].columns]
    R = c['sym'][use].rank(axis=1, pct=True).reindex(columns=GENES)
    d = R.join(c['meta'][['subject', 'passage']]).groupby(['subject', 'passage'], as_index=False).mean(numeric_only=True)
    TR[k] = [(tlog(g.passage), g[GENES].to_numpy(np.float32)) for _, g in d.groupby('subject') if g.passage.nunique() >= 2]
rng = np.random.default_rng(sg.SEED)
def learn(shuffle):
    per = []
    for k, subs in TR.items():
        bs = []
        for t, X in subs:
            tt = rng.permutation(t) if shuffle else t
            tc = tt - tt.mean()
            if (tc ** 2).sum() > 0:
                bs.append(np.nan_to_num((tc[None, :] @ (X - np.nanmean(X, axis=0))).ravel() / (tc ** 2).sum()))
        per.append(np.mean(bs, axis=0))
    return np.mean(per, axis=0)
check = learn(False)
print('Relearned drift vs frozen drift, correlation:', round(float(np.corrcoef(check, BETA)[0, 1]), 5))
neg = []
for rep in range(30):
    r = evaluate(learn(True))
    neg.append(float((r.err_forecast < r.err_carry).mean()))
gate = {
    'donors': len(res), 'cohorts_loaded': len(by),
    'share_better': float(res.forecast_better.mean()),
    'cohorts_better': int((by.err_forecast < by.err_carry).sum()),
    'negative_control_share': float(np.mean(neg)),
    'range80': RANGE80, 'coverage': float(res.in_range.mean()),
    'median_err_forecast': float(res.err_forecast.median()), 'median_err_carry': float(res.err_carry.median()),
    'actual_rose': int(res.actual_rose.sum()),
    'binom_better_p': float(stats.binomtest(int(res.forecast_better.sum()), len(res)).pvalue),
}
gate['passes'] = bool(len(by) >= 2 and gate['share_better'] >= 0.7 and gate['cohorts_better'] >= len(by) - 1 and gate['negative_control_share'] <= 0.6 and gate['coverage'] >= 0.7)
print(json.dumps(gate, indent=1))
json.dump(gate, open(RESULTS / 'forecast_test_gate.json', 'w'), indent=1)
(REPORTS / 'FORECAST_TEST_RESULTS.md').write_text('# Score forecast: independent test\n\nGate: ' + json.dumps(gate, indent=1) + '\n\nBy cohort:\n' + by.round(3).to_string() + '\n\nBy donor:\n' + res.round(3).to_string(index=False) + '\n', encoding='utf-8')

# Long series in GSE9593 donor 1: forecast every later passage from the first sample
if 'GSE9593' in TEST:
    c = TEST['GSE9593']
    R, use = rank_form(c['sym'])
    sig = SIG.reindex(use).dropna(); s_idx = [use.index(g) for g in sig.index]; wr = stats.rankdata(sig.to_numpy())
    score = lambda x: float(np.corrcoef(stats.rankdata(np.asarray(x)[s_idx]), wr)[0, 1])
    b = pd.Series(BETA, index=GENES)[use].to_numpy()
    g = R.join(c['meta']); g = g[g.subject == 'donor1'].sort_values('passage')
    X = g[use].to_numpy(float)
    series = [{'passage': float(p), 'actual': score(x), 'forecast': score(X[0] + b * (tlog(p) - tlog(g.passage.iloc[0]))), 'carry': score(X[0])} for p, x in zip(g.passage, X)]
    print(pd.DataFrame(series).round(3).to_string(index=False))
    pd.DataFrame(series).to_csv(RESULTS / 'forecast_test_long_series.csv', index=False)